# espsdr-demo: analyse a capture log

Opens the `.zip` from the page's **Data logging → Download .zip** and makes the plots the
guided labs ask for. Each section skips itself when the log doesn't have what it needs.

| section | needs |
|---|---|
| 1. Overview | `summary.csv` |
| 2. Waterfall | `spectrum.csv` |
| 3. Channel occupancy | `spectrum.csv` |
| 4. Path loss | `summary.csv` with `beacon_pwr_dbm` (Beacon control → power steps) |
| 5. One capture in time | an `iq_NNNNN.npy` |

Needs `numpy`, `pandas`, `matplotlib`. Set `ZIP` below and run all cells.

In [ ]:
ZIP = 'espsdr-log.zip'   # path to the log you downloaded

In [ ]:
import io, re, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

z = zipfile.ZipFile(ZIP)
names = set(z.namelist())
read = lambda n: z.read(n).decode() if n in names else None
summary = pd.read_csv(io.StringIO(read('summary.csv'))) if 'summary.csv' in names else None
marks = pd.read_csv(io.StringIO(read('marks.csv'))) if 'marks.csv' in names else pd.DataFrame(columns=['t_s', 'iso_time', 'capture', 'label'])
iq_files = sorted(n for n in names if re.fullmatch(r'iq_\d+\.npy', n))

def read_spectrum(text):
    """spectrum.csv -> list of segments. A new header row (bin centres in MHz) appears
    whenever tuning, span or FFT size changed, so each segment has its own frequency axis."""
    segs, cur = [], None
    for line in text.splitlines():
        cells = line.split(',')
        if cells[0] == 'capture':
            cur = {'mhz': np.array(cells[2:], float), 'capture': [], 't': [], 'db': []}
            segs.append(cur)
        elif cur is not None and line:
            cur['capture'].append(int(cells[0])); cur['t'].append(float(cells[1]))
            cur['db'].append(np.array(cells[2:], float))
    for s in segs:
        s['capture'] = np.array(s['capture']); s['t'] = np.array(s['t']); s['db'] = np.array(s['db'])
    return [s for s in segs if len(s['db'])]

spectra = read_spectrum(read('spectrum.csv')) if 'spectrum.csv' in names else []
print(read('README.txt') or '')
print(f"summary rows: {0 if summary is None else len(summary)}, spectrum segments: {len(spectra)}, "
      f"I/Q snapshots: {len(iq_files)}, marks: {len(marks)}")

## 1. Overview: the strongest signal over time

Peak frequency and level for every capture, with your marks as dashed lines.

In [ ]:
if summary is None:
    print('no summary.csv in this log')
else:
    fig, (a1, a2) = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
    a1.plot(summary.t_s, summary.peak_mhz, '.', ms=3); a1.set_ylabel('peak (MHz)')
    a2.plot(summary.t_s, summary.peak_dbfs, '.', ms=3); a2.set_ylabel('peak (dBFS)'); a2.set_xlabel('time (s)')
    for _, m in marks.iterrows():
        for a in (a1, a2): a.axvline(m.t_s, color='m', ls='--', lw=.8)
        a1.annotate(str(m.label), (m.t_s, a1.get_ylim()[1]), rotation=90, va='top', fontsize=8, color='m')
    plt.tight_layout(); plt.show()

## 2. Waterfall

The page's waterfall, rebuilt from `spectrum.csv` — one row per capture, newest at the bottom here.

In [ ]:
for k, s in enumerate(spectra):
    fig, ax = plt.subplots(figsize=(10, 4))
    im = ax.imshow(s['db'], aspect='auto', origin='upper', cmap='viridis',
                   extent=[s['mhz'][0], s['mhz'][-1], s['t'][-1], s['t'][0]],
                   vmin=np.percentile(s['db'], 5), vmax=s['db'].max())
    ax.set_xlabel('MHz'); ax.set_ylabel('time (s)'); ax.set_title(f'segment {k + 1}: {len(s["db"])} captures')
    plt.colorbar(im, label='dBFS'); plt.tight_layout(); plt.show()
if not spectra:
    print('no spectrum.csv in this log: record with spectrum.csv ticked')

## 3. Channel occupancy

Same rule as the page's occupancy panel: in each capture a channel is *busy* when its average power
across its width is more than `THRESHOLD_DB` above that capture's floor (its 20th-percentile bin).
Wi-Fi neighbours overlap, so 1, 6 and 11 are the fair comparison.

In [ ]:
THRESHOLD_DB = 10
WIFI = {ch: (2407 + 5 * ch, 20) for ch in range(1, 14)}
BLE_ADV = {37: (2402, 2), 38: (2426, 2), 39: (2480, 2)}

def occupancy(s, chans):
    floor = np.percentile(s['db'], 20, axis=1)
    out = {}
    for ch, (f, w) in chans.items():
        sel = (s['mhz'] >= f - w / 2) & (s['mhz'] < f + w / 2)
        if f - w / 2 < s['mhz'][0] or f + w / 2 > s['mhz'][-1] or not sel.any():
            continue                                   # not wholly inside this span
        p = 10 * np.log10(np.mean(10 ** (s['db'][:, sel] / 10), axis=1))
        out[ch] = np.mean(p > floor + THRESHOLD_DB)
    return out

for k, s in enumerate(spectra):
    fig, axes = plt.subplots(1, 2, figsize=(11, 3), gridspec_kw={'width_ratios': [3, 1]})
    for ax, (name, chans) in zip(axes, [('Wi-Fi', WIFI), ('BLE advertising', BLE_ADV)]):
        occ = occupancy(s, chans)
        if not occ:
            ax.set_title(f'{name}: none in span'); continue
        ax.bar([str(c) for c in occ], [100 * v for v in occ.values()],
               color=['tab:cyan' if c in (1, 6, 11, 37, 38, 39) else 'tab:gray' for c in occ])
        ax.set_ylim(0, 100); ax.set_ylabel('% of captures busy'); ax.set_title(f'{name} (segment {k + 1})')
    plt.tight_layout(); plt.show()
    key = {c: v for c, v in occupancy(s, WIFI).items() if c in (1, 6, 11)}
    if key:
        print('busiest of 1/6/11:', max(key, key=key.get), {c: f'{100 * v:.0f} %' for c, v in key.items()})

## 4. Path loss

From **Beacon control → power steps** (Hardware AGC off). Received peak against the beacon's transmit
power should rise one dB per dB. Marks like `d=1m` and `d=3m` split the log into distances; the gap between
the fitted lines is the extra path loss. Free space predicts 20·log10(d₂/d₁).

In [ ]:
if summary is None or 'beacon_pwr_dbm' not in summary or summary.beacon_pwr_dbm.isna().all():
    print('no beacon power column: connect the beacon in Beacon control and run power steps while recording')
else:
    df = summary.dropna(subset=['beacon_pwr_dbm']).copy()
    if 'gain_mode' in df and (df.gain_mode == 'HARDWARE').any():
        print('note: some rows were captured with hardware AGC on, which hides level changes')
    # distance from marks like "d=3m" / "d = 1.5 m"; rows before the first one are "?"
    dmarks = [(int(m.capture), float(re.search(r'd\s*=\s*([\d.]+)\s*m', str(m.label)).group(1)))
              for _, m in marks.iterrows() if re.search(r'd\s*=\s*([\d.]+)\s*m', str(m.label))]
    df['distance_m'] = np.nan
    for cap, d in sorted(dmarks):
        df.loc[df.capture >= cap, 'distance_m'] = d
    groups = df.groupby(df.distance_m.fillna(-1))
    fig, ax = plt.subplots(figsize=(7, 4)); fits = {}
    for d, g in groups:
        med = g.groupby('beacon_pwr_dbm').peak_dbfs.median()
        if len(med) < 2: continue
        slope, icpt = np.polyfit(med.index, med.values, 1); fits[d] = icpt + slope * 11
        label = f'{d:g} m' if d >= 0 else 'no distance mark'
        ax.plot(med.index, med.values, 'o', label=f'{label}: {slope:.2f} dB/dB')
        xs = np.array([med.index.min(), med.index.max()]); ax.plot(xs, icpt + slope * xs, '-', alpha=.5)
    ax.set_xlabel('beacon transmit power (dBm)'); ax.set_ylabel('received peak (dBFS, median per step)')
    ax.legend(); ax.grid(alpha=.3); plt.tight_layout(); plt.show()
    ds = sorted(d for d in fits if d > 0)
    for a, b in zip(ds, ds[1:]):
        print(f'{a:g} m -> {b:g} m: measured {fits[a] - fits[b]:.1f} dB more loss, free space predicts {20 * np.log10(b / a):.1f} dB')

## 5. One capture in time

The first raw I/Q snapshot: the envelope |I + jQ| over the burst, like the page's time-domain panel.
The sample rate comes from `summary.csv` when it's there.

In [ ]:
if not iq_files:
    print('no .npy snapshots: record with "raw I/Q as .npy" ticked')
else:
    name = iq_files[0]; cap = int(name[3:8])
    iq = np.load(io.BytesIO(z.read(name))); x = iq[:, 0] + 1j * iq[:, 1]
    rate = float(summary.set_index('capture').rate_hz.get(cap, 80e6)) if summary is not None else 80e6
    t_us = np.arange(len(x)) / rate * 1e6
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot(t_us, np.abs(x), lw=.5); ax.axhline(1, color='r', ls=':', lw=.8)
    ax.set_xlabel('µs'); ax.set_ylabel('|I + jQ|'); ax.set_title(f'{name}: {len(x)} samples at {rate / 1e6:g} MS/s')
    plt.tight_layout(); plt.show()
    print(f'clipped samples: {int(np.sum((np.abs(iq) >= 0.98).any(axis=1)))}')